# Document Ingestion

## Goal
Before retrieval and generation, we need to prepare documents.
Ingestion = load, clean, and chunk documents into pieces
small enough for an embedding model to process.

## Pipeline
Raw Document -> Load -> Clean -> Chunk -> Ready for Embedding

In [2]:
import re
from pathlib import Path

## 1. Load Text Document
We start with plain text - the simplest case.

In [3]:
# Sample document — a short article about machine learning
sample_document = """
Machine learning is a subset of artificial intelligence that enables systems 
to learn and improve from experience without being explicitly programmed. 
It focuses on developing computer programs that can access data and use it 
to learn for themselves.

The process begins with observations or data, such as examples, direct 
experience, or instruction. Machine learning algorithms use computational 
methods to learn information directly from data without relying on a 
predetermined equation as a model.

Deep learning is a subfield of machine learning that uses neural networks 
with many layers. These deep neural networks have revolutionized fields such 
as computer vision, natural language processing, and speech recognition.

Natural language processing (NLP) is another important area of machine learning. 
It enables computers to understand, interpret, and generate human language. 
Applications include sentiment analysis, machine translation, and chatbots.

Reinforcement learning is a type of machine learning where an agent learns 
to make decisions by interacting with an environment. The agent receives 
rewards for correct actions and penalties for incorrect ones.
"""

print(f"Documnet lenght: {len(sample_document)} characters")
print(f"Word count: {len(sample_document.split())}")
print(f"nFirst 200 chars:\n{sample_document[:200]}")

Documnet lenght: 1184 characters
Word count: 166
nFirst 200 chars:

Machine learning is a subset of artificial intelligence that enables systems 
to learn and improve from experience without being explicitly programmed. 
It focuses on developing computer programs tha


## 2.Clean Document
Remove extra whitespace, normalize text.

In [4]:
def clean_text(text: str) -> str:
    """
    Clean raw document text.
    - Remove extra whitespace
    - Normalize newlines
    - Strip leading/trailing spaces
    """
    # Normalize whitespace
    text = re.sub(r"\n+", "\n", text)
    text = re.sub(r" +", " ", text)
    text = text.strip()
    return text

cleaned_doc = clean_text(sample_document)

print(f"Before cleaning: {len(sample_document)} chars")
print(f"After cleaning: {len(cleaned_doc)} chars]")
print(f"\nCleaned document:\n{cleaned_doc[:300]}")

Before cleaning: 1184 chars
After cleaning: 1178 chars]

Cleaned document:
Machine learning is a subset of artificial intelligence that enables systems 
to learn and improve from experience without being explicitly programmed. 
It focuses on developing computer programs that can access data and use it 
to learn for themselves.
The process begins with observations or data, 


## 3. Chunking Strategies
Large documents must be split into smaller chunks.
Each chunk will become one unit in the vector store.

Two strategies:
- Fixed size: split by character count
- Sentence-based: split by sentences

In [8]:
def chunk_by_size(text: str, chunk_size: int = 200, overlap: int = 50) -> list:
    """
    Split text into fixed-size chunks with overlap.
    
    Args:
        text: Input text
        chunk_size: Max characters per chunk
        overlap: Characters shared between consecutive chunks
    
    Returns:
        List of text chunks
    """
    chunks = []
    start = 0
    
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]
        chunks.append(chunk.strip())
        start += chunk_size - overlap
    
    return chunks

def chunk_by_sentences(text: str, sentences_per_chunk: int = 3) -> list:
    """
    Split text into chunks by sentence boundaries.
    
    Args:
        text: Input text
        sentences_per_chunk: Number of sentences per chunk
    
    Returns:
        List of text chunks
    """
    sentences = re.split(r"(?<=[.!?])\s+", text)
    chunks = []
    
    for i in range(0, len(sentences), sentences_per_chunk):
        chunk = ' '.join(sentences[i:i + sentences_per_chunk])
        if chunk.strip():
            chunks.append(chunk.strip())
    
    return chunks

# Test both strategies
size_chunks = chunk_by_size(cleaned_doc, chunk_size=200, overlap=50)
sentence_chunks = chunk_by_sentences(cleaned_doc, sentences_per_chunk=3)

print(f"Fixed-size chunks: {len(size_chunks)}")
print(f"Sentence chunks:   {len(sentence_chunks)}")

print(f"\n--- Fixed-size chunks ---")
for i, chunk in enumerate(size_chunks[:3]):
    print(f"\nChunk {i+1} ({len(chunk)} chars):\n{chunk}")

print(f"\n--- Sentence chunks ---")
for i, chunk in enumerate(sentence_chunks[:3]):
    print(f"\nChunk {i+1}:\n{chunk}")

Fixed-size chunks: 8
Sentence chunks:   4

--- Fixed-size chunks ---

Chunk 1 (200 chars):
Machine learning is a subset of artificial intelligence that enables systems 
to learn and improve from experience without being explicitly programmed. 
It focuses on developing computer programs that

Chunk 2 (200 chars):
. 
It focuses on developing computer programs that can access data and use it 
to learn for themselves.
The process begins with observations or data, such as examples, direct 
experience, or instructi

Chunk 3 (200 chars):
such as examples, direct 
experience, or instruction. Machine learning algorithms use computational 
methods to learn information directly from data without relying on a 
predetermined equation as a m

--- Sentence chunks ---

Chunk 1:
Machine learning is a subset of artificial intelligence that enables systems 
to learn and improve from experience without being explicitly programmed. It focuses on developing computer programs that can access data and use it 

## 4. Key Observations

| Strategy | Chunks | Pros | Cons |
|----------|--------|------|------|
| Fixed-size | 8 | Simple, predictable size | Cuts mid-sentence |
| Sentence-based | 4 | Preserves meaning | Variable chunk size |

## Key Insight
Chunking strategy directly affects retrieval quality.
A chunk that cuts mid-sentence loses context — the embedding will be noisy.
Sentence-based chunking preserves semantic units.

For this project we use sentence-based chunking.

## Next
Embed the chunks and store them in a vector database.
-> 02_retrieval.ipynb